# Data Preprocessing and Feature Engineering

Prepare the housing data using the saved train/test split and decisions documented during EDA. Imputation, encoding, and scaling will be fitted within training folds to prevent data leakage. The test set will remain reserved for final evaluation.

## 1. Import Required Libraries

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load the Saved Training Split

Reconstruct the training dataset using the previously saved row indices. Preserve the original row order of the raw dataset so that the saved indices identify the correct records.

In [4]:
project_root = Path.cwd().parent

df = pd.read_csv(project_root / "data/raw/melb_data.csv")
train_indices = pd.read_csv(
    project_root / "data/processed/train_indices.csv"
)["row_index"]

train_df = df.loc[train_indices].copy()

train_df.shape

(10864, 21)

### 3.1 Treat Zero Building Area as Missing

EDA identified 15 zero building-area values. Treat these as unavailable measurements while retaining the property records.

In [5]:
clean_train = train_df.copy()

clean_train["BuildingArea"] = clean_train["BuildingArea"].replace(0, np.nan)

clean_train["BuildingArea"].isna().sum()

np.int64(5144)

### 3.2 Treat Zero Land Size in Houses as Missing

Treat zero `Landsize` values for houses as unavailable information. Retain zeros for units and townhouses because shared-land recording conventions may apply.

In [6]:
mask = (clean_train["Type"] == "h") & (clean_train["Landsize"] == 0)

clean_train.loc[mask, "Landsize"] = np.nan

clean_train["Landsize"].isna().sum()

np.int64(138)

### 3.3 Treat Zero Bathroom Counts as Missing

Treat zero bathroom counts as unavailable information based on the EDA findings. Retain all property records.

In [7]:
clean_train["Bathroom"] = clean_train["Bathroom"].replace(0, np.nan)

clean_train["Bathroom"].isna().sum()

np.int64(25)

The 25 zero bathroom counts were treated as missing because they may represent unavailable information. All property records were retained; these missing values will be handled during preprocessing.

### 3.4 Treat Zero Bedroom Counts as Missing

Zero values in `Bedroom2` are treated as unavailable bedroom counts for this analysis. All records are retained, and the original `Rooms` values remain unchanged.

In [8]:
clean_train["Bedroom2"] = clean_train["Bedroom2"].replace(0, np.nan)

clean_train["Bedroom2"].isna().sum()

np.int64(12)

The 12 zero values in `Bedroom2` were treated as missing under the documented cleaning assumption. All records were retained, and no values were copied from `Rooms`.

### 3.5 Review the Extreme Bedroom Count

EDA identified a property with 3 rooms but 20 recorded bedrooms. Inspect this discrepancy before deciding how to handle it during preprocessing.

In [9]:
clean_train.loc[
    clean_train["Bedroom2"] == 20,
    ["Rooms", "Bedroom2", "Bathroom", "Type", "BuildingArea"]
]

,Rooms,Bedroom2,Bathroom,Type,BuildingArea
7404,3,20.0,1.0,h,NaN


The record contains 3 rooms, 20 bedrooms, and 1 bathroom, with building area unavailable. Given this substantial inconsistency, the bedroom count is treated as missing as a documented cleaning assumption. The property is retained without guessing a replacement value.

In [10]:
clean_train.loc[clean_train["Bedroom2"] == 20, "Bedroom2"] = np.nan

clean_train["Bedroom2"].isna().sum()

np.int64(13)

`Bedroom2` now contains 13 missing values: 12 zero counts and one inconsistent count of 20. All property records were retained.

### 3.6 Parse Sale Dates

Convert sale dates to datetime using the dataset's day/month/year format. These dates will support construction-year consistency checks and the calculation of property age at sale.

In [11]:
clean_train["Date"] = pd.to_datetime(
    clean_train["Date"], format="%d/%m/%Y", errors="coerce"
)

clean_train["Date"].isna().sum()

np.int64(0)

**Observation**

All 10,864 sale dates were successfully parsed using the day/month/year format. No missing dates remained after conversion.

### 3.7 Calculate Property Age at Sale

Calculate approximate property age by subtracting the construction year from the sale year. Missing construction years produce missing ages. Negative ages are identified for review.

In [12]:
clean_train["Age_At_Sale"] = (
    clean_train["Date"].dt.year - clean_train["YearBuilt"]
)

(clean_train["Age_At_Sale"] < 0).sum()

np.int64(6)

**Observation**

Six properties have a recorded construction year later than their sale year, producing negative ages. These may reflect recording errors or planned construction, but the available data cannot confirm the cause. Their calculated ages require handling before use in modeling.

### 3.8 Treat Negative Property Ages as Missing

Negative ages are treated as missing because they cannot represent completed property age at sale. All records and original construction years are retained without guessing replacement ages.

In [13]:
clean_train.loc[
    clean_train["Age_At_Sale"] < 0, "Age_At_Sale"
] = np.nan

clean_train["Age_At_Sale"].isna().sum()

np.int64(4292)

**Observation**

`Age_At_Sale` contains 4,292 missing values: 4,286 from missing construction years and 6 from negative calculated ages. All property records and original `YearBuilt` values were retained. Valid zero ages, representing construction and sale in the same year, remain unchanged.

### 3.9 Review Missing Values After Cleaning

Summarize missing values after applying the documented cleaning rules and creating `Age_At_Sale`. This check verifies the resulting counts before imputation.

In [14]:
missing_after_cleaning = clean_train.isna().sum()

missing_after_cleaning[missing_after_cleaning > 0]

Bedroom2          13
Bathroom          25
Car               50
Landsize         138
BuildingArea    5144
YearBuilt       4286
CouncilArea     1090
Age_At_Sale     4292
dtype: int64

**Observation**

Missing-value counts match the documented cleaning rules. `BuildingArea` has 5,144 missing values, while `Age_At_Sale` has 4,292, including six negative ages treated as missing. Additional missing values occur in `Bedroom2`, `Bathroom`, and house `Landsize` following cleaning. Existing missing values in `Car`, `YearBuilt`, and `CouncilArea` remain unchanged. Imputation has not yet been applied.

### 3.10 Verify Record and Target Preservation

Confirm that cleaning preserved the training records, their index order, and the original target values.

In [15]:
print("Shape after cleaning:", clean_train.shape)
print("Same row indices:", clean_train.index.equals(train_df.index))
print("Price unchanged:", clean_train["Price"].equals(train_df["Price"]))

Shape after cleaning: (10864, 22)
Same row indices: True
Price unchanged: True


**Observation**

The cleaned training data contains 10,864 records and 22 columns. Row indices and their order match the original training split, and all `Price` values remain unchanged. The additional column is the derived `Age_At_Sale` feature.

### 3.11 Save the Cleaned Training Data

Save the cleaned training data with its original row indices. Missing values are retained because imputation, encoding, and scaling will be fitted within training folds during modeling. The raw dataset remains unchanged.

In [17]:
clean_train.to_csv(
    project_root / "data/processed/train_cleaned.csv",
    index=True,
    index_label="row_index"
)

### 3.12 Verify the Saved Training Data

Reload the saved CSV to confirm its shape, original row indices, and retained missing values.

In [18]:
saved_train = pd.read_csv(
    project_root / "data/processed/train_cleaned.csv",
    index_col="row_index",
    parse_dates=["Date"]
)

print("Saved shape:", saved_train.shape)
print("Same row indices:", saved_train.index.equals(clean_train.index))
print("Same missing counts:",
      saved_train.isna().sum().equals(clean_train.isna().sum()))

Saved shape: (10864, 22)
Same row indices: True
Same missing counts: True


**Observation**

The saved training data was successfully reloaded with 10,864 records and 22 columns. Original row indices and per-column missing-value counts match the cleaned DataFrame, confirming that these were preserved during export.